# NFL Play Transformer -- Colab setup

Before running any cells: **Runtime -> Change runtime type -> T4 GPU**, then Save.

T4 is enough here -- the model is ~1.2M parameters, nowhere near needing an A100/L4/H100.

**One-time setup (do this once, outside this notebook):**
1. Create a GitHub Personal Access Token with `repo` scope: https://github.com/settings/tokens -> Generate new token (classic) -> check `repo`.
2. In this Colab session, click the key icon (Secrets) in the left sidebar.
3. Add a new secret named `GITHUB_TOKEN` with that token as the value, and toggle notebook access on.

This notebook never sees or stores the token in its own text -- it's pulled from Colab's Secrets manager at runtime.

## 1. Confirm the GPU is attached

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Clone the private repo (using the GITHUB_TOKEN secret)

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/havishs/nfl-play-transformer.git
%cd nfl-play-transformer


## 3. Install dependencies

torch/pandas/numpy/pyarrow are already present in Colab (with CUDA-enabled torch).
nfl_data_py pins pandas<2.0/numpy<1.0, which is stale -- installed with `--no-deps`
so it doesn't try to downgrade Colab's own pandas/numpy.

In [ ]:
!pip install -q nfl_data_py==0.3.3 --no-deps
!pip install -q requests==2.34.2 appdirs==1.4.4

import torch
print('cuda available:', torch.cuda.is_available())
print('device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


## 4. Mount Google Drive

Colab's local disk is wiped when the runtime disconnects/recycles. The dataset
cache (~1GB) and checkpoint are worth keeping across sessions, so this mounts
Drive and we'll copy results there after training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/nfl-play-transformer'
os.makedirs(DRIVE_DIR, exist_ok=True)


## 5. Fetch data

`data/` is gitignored (not in the repo), so this re-fetches the 2018-2023
seasons via nfl_data_py -- same script already validated locally. If you'd
rather not re-download, you can instead upload the local `data/` folder to
`DRIVE_DIR/data` once and symlink it here -- ask if you want that path instead.

In [ ]:
%cd pipeline
!python fetch_data.py 2018 2019 2020 2021 2022 2023


## 6. Restore a cached dataset build from Drive, if one exists

Skips the ~18-20 min `iterrows()` rebuild on repeat runs. First run this
session: nothing to restore yet, this cell just no-ops.

In [ ]:
import glob, shutil
for f in glob.glob(f'{DRIVE_DIR}/dataset_cache_*.pkl'):
    shutil.copy(f, '.')
    print('restored', f)


## 7. Train

Device selection in train.py already checks `cuda` after `mps`, so this picks
up the T4 automatically -- no code changes needed for the GPU switch itself.

In [ ]:
!python train.py


## 8. Persist results to Drive

In [ ]:
!cp checkpoint.pt {DRIVE_DIR}/
!cp dataset_cache_*.pkl {DRIVE_DIR}/
print('saved to', DRIVE_DIR)
